# Stage 2 Launcher — Joint NER + Sentiment training with global-attention dropout

**What this does:** loads Stage 1's `best_cls_only.pt` (epoch 2, CLS-only NER F1 = 0.5841) and continues training with joint NER + sentiment loss using the curriculum schedule. Same global-attention dropout (p=0.3) continues — no warmup needed since Stage 1 already established the regime.

**Why `best_cls_only.pt` and not `best_model.pt`?**
Stage 1's `best_model.pt` (epoch 4) has entity-aware F1=0.77 but CLS-only F1=0.47.
Stage 1's `best_cls_only.pt` (epoch 2) has entity-aware F1=0.76 and CLS-only F1=0.58.
We care about the inference (CLS-only) regime, so we use the CLS-only-best checkpoint as Stage 2's starting point.

**Estimated time:** ~7-8 hours on the 95 GB GPU at batch_size=20 with gradient checkpointing.

**Watch:** end-of-epoch `Validation [CLS-only]: ner_f1=...` — this is the gating signal. We want it to **stay above 0.40** through Stage 2. The risk is that the joint sentiment loss pulls the encoder away from the NER-friendly regime.

In [ ]:
# 1. Mount Drive & check GPU
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Set project path & locate Stage 1 checkpoint
import os
PROJECT_PATH = "/content/drive/MyDrive/entity_sentiment_model_pipeline"
assert os.path.exists(PROJECT_PATH), f"Not found: {PROJECT_PATH}"
assert os.path.exists(f"{PROJECT_PATH}/scripts/training/train_two_stage.py"), "train_two_stage.py not found!"

# Stage 1 input — load the CLS-only-best checkpoint (best for e2e regime)
STAGE1_CKPT = f"{PROJECT_PATH}/checkpoints/stage1_ner_large_v2/best_cls_only.pt"
assert os.path.exists(STAGE1_CKPT), f"Stage 1 checkpoint missing: {STAGE1_CKPT}"
size_gb = os.path.getsize(STAGE1_CKPT) / 1e9
print(f"Project       : {PROJECT_PATH}")
print(f"Stage 1 input : {STAGE1_CKPT}  ({size_gb:.2f} GB)")

# Local-first save: checkpoints write to /content/, sync to Drive at the end
LOCAL_CKPT_DIR = "/content/stage2_local"
DRIVE_CKPT_DIR = f"{PROJECT_PATH}/checkpoints/stage2_joint_large_v2"
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f"Local ckpts   : {LOCAL_CKPT_DIR}")
print(f"Drive ckpts   : {DRIVE_CKPT_DIR}")

In [ ]:
# 3. Install deps
!pip install -q transformers torch torchvision torchaudio
!pip install -q pytorch-crf

In [ ]:
# 4. Run Stage 2
#    Key differences from Stage 1:
#      --stage 2                                     ← joint NER+sentiment with curriculum
#      --stage1_checkpoint <best_cls_only.pt path>   ← load CLS-best, not training-best
#      --batch_size 20                               ← Stage 2 has extra V1 sentiment head activations
#      --stage2_dropout_warmup_frac 0.0              ← Stage 1 already established the regime
#
#    Memory: with gradient_checkpointing + batch=20, should peak ~70-75 GB on the 95 GB GPU.
#    If first few batches show < 60 GB usage, can bump to batch=24 in a future run.
#
#    Gating signal: Validation [CLS-only]: ner_f1=... each epoch.
#                   Want it to STAY above 0.40 through training.

!cd {PROJECT_PATH} && python scripts/training/train_two_stage.py \
    --stage 2 \
    --batch_size 24 \
    --stage2_epochs 5 \
    --stage2_lr 1e-5 \
    --global_attn_dropout_prob 0.3 \
    --stage2_dropout_warmup_frac 0.0 \
    --gradient_checkpointing \
    --stage1_checkpoint {STAGE1_CKPT} \
    --stage2_checkpoint_dir {LOCAL_CKPT_DIR}

In [ ]:
# 5. Show local checkpoints (these are safe to copy to Drive)
import os
print(f"Local checkpoints in {LOCAL_CKPT_DIR}:")
for f in sorted(os.listdir(LOCAL_CKPT_DIR)):
    path = os.path.join(LOCAL_CKPT_DIR, f)
    if os.path.isfile(path):
        size = os.path.getsize(path) / 1e6
        print(f"  {f:40s} {size:8.1f} MB")

# Latest training log
import glob
logs = sorted(glob.glob(f"{PROJECT_PATH}/logs/two_stage_training_*.log"))
if logs:
    print(f"\nLatest log: {logs[-1]} (last 50 lines)")
    with open(logs[-1]) as fh:
        for line in fh.readlines()[-50:]:
            print(f"  {line.rstrip()}")

In [ ]:
# 6. Sync local checkpoints -> Drive (local-first pattern)
#    Same pattern as Stage 1 — Drive FUSE flush + remount to ensure writes land.
import shutil, os, time

src_dir = LOCAL_CKPT_DIR
dst_dir = DRIVE_CKPT_DIR
os.makedirs(dst_dir, exist_ok=True)

print(f"Syncing {src_dir} -> {dst_dir}")
for f in sorted(os.listdir(src_dir)):
    src = os.path.join(src_dir, f)
    dst = os.path.join(dst_dir, f)
    if os.path.isfile(src):
        try:
            shutil.copy2(src, dst)
            size = os.path.getsize(dst) / 1e6
            print(f"  {f:40s} -> Drive OK  ({size:.1f} MB)")
        except Exception as e:
            print(f"  {f:40s} -> FAILED: {e}")

# Flush Drive buffer to force FUSE to actually write
from google.colab import drive
print('\nFlushing Drive...')
drive.flush_and_unmount()
time.sleep(3)
drive.mount('/content/drive')
time.sleep(5)
print('Drive remounted.')

In [ ]:
# 7. Verify the checkpoints actually landed on Drive
import os
print(f"Drive checkpoints in {DRIVE_CKPT_DIR}:")
if os.path.exists(DRIVE_CKPT_DIR):
    for f in sorted(os.listdir(DRIVE_CKPT_DIR)):
        path = os.path.join(DRIVE_CKPT_DIR, f)
        if os.path.isfile(path):
            size = os.path.getsize(path) / 1e6
            print(f"  {f:40s} {size:8.1f} MB")
else:
    print(f"  Directory not visible yet — Drive FUSE may still be reconnecting.")
    print(f"  Local copies are at: {LOCAL_CKPT_DIR}")

In [ ]:
# 8. (Optional) Terminate runtime to stop billing
#    Only run when cell 7 has confirmed the Drive sync.
from google.colab import runtime
runtime.unassign()